# 📦 SmartStock – Product Performance & Profitability Analysis
### Best Sellers, Slow-Moving Inventory & Product Profit Margins
**Objective**: Identify catalog champions, slow-moving SKUs, gross margin leaders, and pricing markup efficiency.


## 1. Load Product and Sale Line Item Data


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = os.path.join('..', 'datasets')
df_products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
df_items = pd.read_csv(os.path.join(DATA_DIR, 'sale_items.csv'))

print(f"Loaded {len(df_products)} catalog products and {len(df_items)} line items.")
df_products.head(3)


## 2. Best-Selling Products (By Volume and Revenue)
Rank top 10 products driving top-line revenue and total checkout volume.


In [ ]:
prod_agg = df_items.groupby(['product_id', 'product_name', 'category_name']).agg(
    units_sold=('quantity', 'sum'),
    revenue_generated=('subtotal', 'sum'),
    total_cogs=('cogs', 'sum'),
    total_profit=('gross_profit', 'sum')
).reset_index()

prod_agg['gross_margin_pct'] = (prod_agg['total_profit'] / prod_agg['revenue_generated']) * 100

# Top 10 by Units Sold
top_units = prod_agg.sort_values(by='units_sold', ascending=False).head(10)

# Top 10 by Revenue
top_rev = prod_agg.sort_values(by='revenue_generated', ascending=False).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.barh(top_units['product_name'][::-1], top_units['units_sold'][::-1], color='#3b82f6')
ax1.set_title('Top 10 Best Sellers by Unit Volume', fontweight='bold', fontsize=12)
ax1.set_xlabel('Units Sold')

ax2.barh(top_rev['product_name'][::-1], top_rev['revenue_generated'][::-1], color='#10b981')
ax2.set_title('Top 10 Products by Total Revenue', fontweight='bold', fontsize=12)
ax2.set_xlabel('Revenue ($)')
ax2.xaxis.set_major_formatter('${x:,.0f}')

plt.tight_layout()
plt.show()


## 3. Slow-Moving Products & Velocity Analysis
Identify catalog items with the lowest sales frequency or sluggish movement relative to on-hand inventory.


In [ ]:
# Merge with product catalog stock levels
prod_velocity = pd.merge(df_products, prod_agg, on='product_id', how='left')
prod_velocity['units_sold'] = prod_velocity['units_sold'].fillna(0)
prod_velocity['revenue_generated'] = prod_velocity['revenue_generated'].fillna(0)

# Sort bottom products by volume
slow_movers = prod_velocity.sort_values(by='units_sold', ascending=True).head(8)

display(slow_movers[['sku', 'name', 'category_name', 'current_stock', 'units_sold', 'revenue_generated']])

plt.figure(figsize=(12, 5))
plt.barh(slow_movers['name'][::-1], slow_movers['units_sold'][::-1], color='#f97316')
plt.title('Bottom Slow-Moving SKUs (Lowest Sales Velocity)', fontweight='bold', fontsize=13)
plt.xlabel('Total Units Sold in 180 Days')
plt.tight_layout()
plt.show()


## 4. Product Profitability & Margin Matrix
Analyze gross profit generation vs. sales volume across categories to determine high-margin superstars vs. low-margin drivers.


In [ ]:
plt.figure(figsize=(12, 7))

scatter = plt.scatter(
    prod_agg['units_sold'], 
    prod_agg['gross_margin_pct'],
    s=prod_agg['revenue_generated'] / 18, 
    c=prod_agg['total_profit'], 
    cmap='viridis', 
    alpha=0.75, 
    edgecolors='black', 
    linewidth=1
)

plt.title('Product Profitability Matrix (Bubble Size = Total Revenue)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Units Sold (Volume)', fontweight='semibold')
plt.ylabel('Gross Margin (%)', fontweight='semibold')
cbar = plt.colorbar(scatter)
cbar.set_label('Total Gross Profit ($)', fontweight='semibold')

# Annotate key products
for _, r in prod_agg.nlargest(5, 'total_profit').iterrows():
    plt.annotate(r['product_name'][:18] + '...', (r['units_sold'] + 2, r['gross_margin_pct'] + 0.3), fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Most vs. Least Profitable Products Table


In [ ]:
top_profitable = prod_agg.sort_values(by='total_profit', ascending=False).head(5)
least_profitable = prod_agg.sort_values(by='total_profit', ascending=True).head(5)

print("🏆 Top 5 Most Profitable Products:")
display(top_profitable[['product_name', 'category_name', 'units_sold', 'revenue_generated', 'total_profit', 'gross_margin_pct']])

print("\n⚠️ Bottom 5 Least Profitable Products:")
display(least_profitable[['product_name', 'category_name', 'units_sold', 'revenue_generated', 'total_profit', 'gross_margin_pct']])
